# Pneumonia Detection v5 — NIH ChestX-ray14
### MC Dropout + Uncertainty-Aware Triage + Three-Way Calibration
**Why NIH over Kermany:**
- 112,120 real-world X-rays from 30,805 unique patients (vs 5,216 curated images)
- Multi-label, noisy, genuinely ambiguous cases → MC Dropout uncertainty is meaningful
- Multi-site hospital data → uncertainty spikes on harder, real-world images
- Standard benchmark for published chest X-ray work post-2017

**Key changes from v4:**
- Dataset: NIH ChestX-ray14 (pneumonia vs no-finding)
- Dropout inserted at **every DenseBlock** in the backbone (not just the head)
- Patient-level train/test split (prevents data leakage across patient scans)
- Uncertainty signal expected to work correctly on this dataset
---

## Cell 1 — Setup

In [ ]:
import torch, os
print("GPU:", torch.cuda.is_available())
!pip install -q opencv-python-headless scikit-learn matplotlib pandas

## Cell 2 — Download NIH dataset
The NIH dataset is ~45GB. On Kaggle it's pre-cached. On Colab use the kaggle API.

In [ ]:
# ── On Kaggle (recommended — dataset is pre-cached) ───────────────────────
# Add the dataset via the Data tab: nih-chest-xrays/data
# Then set NIH_DIR below.

# ── On Google Colab ────────────────────────────────────────────────────────
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d nih-chest-xrays/data --unzip -p /content/nih

NIH_DIR = "/kaggle/input/data"          # adjust if on Colab
IMG_DIR  = os.path.join(NIH_DIR, "images")
CSV_PATH = os.path.join(NIH_DIR, "Data_Entry_2017.csv")

print("CSV exists :", os.path.exists(CSV_PATH))
print("Image count:", len(os.listdir(IMG_DIR)) if os.path.exists(IMG_DIR) else "NOT FOUND")

## Cell 3 — Load & filter NIH dataset
**Binary task:** Pneumonia (1) vs No Finding (0).
**Patient-level split:** a patient's scans must not appear in both train and test.
This is critical for NIH — patients have multiple scans, and random splitting
leaks patient identity across splits, inflating test performance.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv(CSV_PATH)
print("Total images :", len(df))
print("Columns      :", df.columns.tolist())

# ── Binary filter: Pneumonia vs No Finding ─────────────────────────────────
pneumonia = df[df["Finding Labels"].str.contains("Pneumonia", na=False)].copy()
normal    = df[df["Finding Labels"] == "No Finding"].copy()

pneumonia["label"] = 1
normal["label"]    = 0

# Subsample normal to 3:1 ratio (no-finding >> pneumonia in NIH)
normal_sample = normal.sample(n=min(len(pneumonia)*3, len(normal)), random_state=42)
data = pd.concat([pneumonia, normal_sample]).reset_index(drop=True)

print(f"\nPneumonia : {len(pneumonia)}")
print(f"Normal    : {len(normal_sample)}")
print(f"Total     : {len(data)}")

# ── Patient-level split (CRITICAL for NIH) ─────────────────────────────────
# A patient_id column is 'Patient ID' in NIH CSV
patient_col = "Patient ID"
all_patients = data[patient_col].unique()

tr_patients, tmp_patients = train_test_split(
    all_patients, test_size=0.30, random_state=42)
val_patients, test_patients = train_test_split(
    tmp_patients, test_size=0.50, random_state=42)

train_df = data[data[patient_col].isin(tr_patients)].reset_index(drop=True)
val_df   = data[data[patient_col].isin(val_patients)].reset_index(drop=True)
test_df  = data[data[patient_col].isin(test_patients)].reset_index(drop=True)

print(f"\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Train pneumonia: {train_df.label.sum()} | normal: {(train_df.label==0).sum()}")
print(f"Test  pneumonia: {test_df.label.sum()}  | normal: {(test_df.label==0).sum()}")

## Cell 4 — PyTorch Dataset class

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class NIHChestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        # Only keep rows where the image file actually exists
        df = df.copy()
        df["exists"] = df["Image Index"].apply(
            lambda x: os.path.exists(os.path.join(img_dir, x)))
        self.df        = df[df["exists"]].reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform
        dropped = len(df) - len(self.df)
        if dropped > 0:
            print(f"  Dropped {dropped} missing images")

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(
            self.img_dir, row["Image Index"])).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(row["label"])

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

print("Building datasets...")
train_ds = NIHChestDataset(train_df, IMG_DIR, train_tf)
val_ds   = NIHChestDataset(val_df,   IMG_DIR, val_tf)
test_ds  = NIHChestDataset(test_df,  IMG_DIR, val_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

## Cell 5 — Class imbalance

In [ ]:
from collections import Counter
import torch

counts = Counter(train_ds.df["label"].tolist())
print(f"Normal: {counts[0]}  Pneumonia: {counts[1]}")
print(f"Ratio : {counts[1]/counts[0]:.3f}x")

device     = "cuda" if torch.cuda.is_available() else "cpu"
pos_weight = torch.tensor([counts[1] / counts[0]]).to(device)
print(f"pos_weight = {pos_weight.item():.3f}")

## Cell 6 — MC Dropout model with backbone dropout
**Key change from v4:** Dropout is inserted after EVERY DenseBlock in the backbone,
not just at the classification head. This gives 5 stochastic dropout points across
the network, producing meaningful variance across MC passes.

With only head dropout (v4): ~98% of network is deterministic → near-zero uncertainty
With backbone dropout (v5):  ~60% of network is stochastic → real uncertainty spread

In [ ]:
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F

class MCDropoutDenseNet(nn.Module):
    def __init__(self, dropout_p=0.3):
        super().__init__()
        base = models.densenet121(weights="IMAGENET1K_V1")

        # ── Extract individual DenseNet components ─────────────────────────
        self.conv0    = base.features.conv0
        self.norm0    = base.features.norm0
        self.relu0    = nn.ReLU(inplace=True)
        self.pool0    = base.features.pool0

        self.block1   = base.features.denseblock1
        self.trans1   = base.features.transition1
        self.drop1    = nn.Dropout2d(p=dropout_p)   # after block1

        self.block2   = base.features.denseblock2
        self.trans2   = base.features.transition2
        self.drop2    = nn.Dropout2d(p=dropout_p)   # after block2

        self.block3   = base.features.denseblock3
        self.trans3   = base.features.transition3
        self.drop3    = nn.Dropout2d(p=dropout_p)   # after block3

        self.block4   = base.features.denseblock4
        self.norm5    = base.features.norm5
        self.drop4    = nn.Dropout2d(p=dropout_p)   # after block4

        self.drop_fc  = nn.Dropout(p=dropout_p)     # before classifier
        self.classifier = nn.Linear(1024, 1)

    def forward(self, x):
        x = self.pool0(self.relu0(self.norm0(self.conv0(x))))

        x = self.drop1(self.trans1(self.block1(x)))
        x = self.drop2(self.trans2(self.block2(x)))
        x = self.drop3(self.trans3(self.block3(x)))
        x = self.drop4(F.relu(self.norm5(self.block4(x))))

        x = F.adaptive_avg_pool2d(x, (1,1))
        x = torch.flatten(x, 1)
        x = self.drop_fc(x)
        return self.classifier(x)

model = MCDropoutDenseNet(dropout_p=0.3).to(device)
n_dropout = sum(1 for m in model.modules() if isinstance(m, (nn.Dropout, nn.Dropout2d)))
print(f"{device} | params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Dropout layers: {n_dropout} (was 2 in v4)")

## Cell 7 — Training with early stopping

In [ ]:
import torch.optim as optim

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2, verbose=True)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.float().to(device)
            if train: optimizer.zero_grad()
            out  = model(imgs).squeeze()
            loss = criterion(out, labels)
            if train: loss.backward(); optimizer.step()
            total_loss += loss.item()
            preds   = (torch.sigmoid(out) > 0.5).long()
            correct += (preds == labels.long()).sum().item()
            total   += len(labels)
    return total_loss / len(loader), correct / total

best_val_loss, patience_ctr, patience = float("inf"), 0, 4

for epoch in range(25):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    scheduler.step(vl_loss)
    print(f"Ep {epoch+1:02d} | tr={tr_loss:.4f}/{tr_acc:.4f} | val={vl_loss:.4f}/{vl_acc:.4f}")

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        patience_ctr  = 0
        torch.save(model.state_dict(), "best_model_v5.pth")
        print(f"         ✓ Saved (val_loss={best_val_loss:.4f})")
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            print(f"Early stop at epoch {epoch+1}")
            break

model.load_state_dict(torch.load("best_model_v5.pth"))
print("\nBest model loaded.")

## Cell 8 — MC Dropout inference (T=50)
Uses enable_mc_dropout() — freezes BatchNorm (eval), keeps all 5 Dropout layers active (train).
With backbone dropout, T=50 is sufficient for stable estimates.

In [ ]:
def enable_mc_dropout(model):
    """Freeze BatchNorm in eval(), re-enable all Dropout layers in train()."""
    model.eval()
    for m in model.modules():
        if isinstance(m, (nn.Dropout, nn.Dropout2d)):
            m.train()
    return model

def mc_predict(model, loader, T=50):
    enable_mc_dropout(model)
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            passes = torch.stack([
                torch.sigmoid(model(imgs)) for _ in range(T)
            ])
            all_probs.append(passes.cpu().numpy())
            all_labels.append(labels.numpy())

    probs  = np.concatenate(all_probs,  axis=1).squeeze(-1)  # (T, N)
    labels = np.concatenate(all_labels)
    return probs.mean(axis=0), probs.var(axis=0), labels

mean_p, var_p, true_labels = mc_predict(model, test_loader, T=50)
print(f"Samples           : {len(mean_p)}")
print(f"Uncertainty range : {var_p.min():.6f}  to  {var_p.max():.6f}")
print(f"Mean uncertainty  : {var_p.mean():.6f}")
print(f"Std  uncertainty  : {var_p.std():.6f}")
print()
# Key diagnostic: uncertainty(incorrect) should > uncertainty(correct)
preds_05 = (mean_p > 0.5).astype(int)
print(f"Uncertainty correct   : {var_p[preds_05==true_labels].mean():.6f}")
print(f"Uncertainty incorrect : {var_p[preds_05!=true_labels].mean():.6f}")
print("Signal direction OK :" , var_p[preds_05!=true_labels].mean() > var_p[preds_05==true_labels].mean())

## Cell 9 — Optimal threshold via Youden's J

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, f1_score, classification_report

fpr, tpr, thresholds = roc_curve(true_labels, mean_p)
youdens_j   = tpr + (1 - fpr) - 1
best_thresh = thresholds[np.argmax(youdens_j)]

print(f"Optimal threshold: {best_thresh:.3f}")
print()
print("Default threshold (0.5):")
print(classification_report(true_labels, (mean_p>0.5).astype(int),
    target_names=["Normal","Pneumonia"]))
print(f"Optimal threshold ({best_thresh:.3f}):")
preds_opt = (mean_p > best_thresh).astype(int)
print(classification_report(true_labels, preds_opt,
    target_names=["Normal","Pneumonia"]))

f1s = [f1_score(true_labels,(mean_p>t).astype(int),zero_division=0) for t in thresholds]
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(thresholds, youdens_j[:len(thresholds)], label="Youden's J", color="#185FA5")
ax.plot(thresholds, f1s, label="F1", color="#639922")
ax.axvline(best_thresh, color="red", linestyle="--",
           label=f"Optimal = {best_thresh:.3f}")
ax.set_xlabel("Threshold"); ax.set_ylabel("Score")
ax.set_title("Threshold selection — Youden's J vs F1")
ax.legend(); plt.tight_layout()
plt.savefig("threshold_selection.png", dpi=150); plt.show()

## Cell 10 — Three-way calibration study
Uncalibrated → Temperature Scaling → Isotonic Regression.
With NIH's noisier labels, expect ECE to be higher than Kermany
but isotonic regression should still bring it below 0.10.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression

def ece(probs, labels, n_bins=10):
    bins = np.linspace(0,1,n_bins+1); err = 0.0
    for i in range(n_bins):
        mask = (probs>=bins[i])&(probs<bins[i+1])
        if mask.sum()==0: continue
        err += mask.sum()*abs(labels[mask].mean()-probs[mask].mean())
    return err/len(probs)

ece_raw = ece(mean_p, true_labels)

# Temperature scaling (uses model.eval() logits — clean)
def get_logits(model, loader):
    model.eval(); logits, labs = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            logits.append(model(imgs.to(device)).squeeze().cpu())
            labs.append(labels)
    return torch.cat(logits), torch.cat(labs).float()

val_logits, val_labs   = get_logits(model, val_loader)
test_logits, _         = get_logits(model, test_loader)

class TempScaler(nn.Module):
    def __init__(self): super().__init__(); self.T = nn.Parameter(torch.ones(1))
    def forward(self, x): return x / self.T

scaler  = TempScaler()
ts_opt  = optim.LBFGS([scaler.T], lr=0.01, max_iter=50)
ts_crit = nn.BCEWithLogitsLoss()
def ts_step():
    ts_opt.zero_grad()
    loss = ts_crit(scaler(val_logits), val_labs)
    loss.backward(); return loss
ts_opt.step(ts_step)
T_learned  = scaler.T.item()
temp_probs = torch.sigmoid(scaler(test_logits)).detach().numpy()
ece_temp   = ece(temp_probs, true_labels)

# Isotonic regression
val_probs = torch.sigmoid(val_logits).numpy()
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(val_probs, val_labs.numpy())
iso_probs = iso.predict(mean_p)
ece_iso   = ece(iso_probs, true_labels)

print("="*52)
print(f"  Uncalibrated          ECE = {ece_raw:.4f}")
print(f"  Temperature scaling   ECE = {ece_temp:.4f}  (T={T_learned:.3f})")
print(f"  Isotonic regression   ECE = {ece_iso:.4f}")
print("="*52)

# Three-panel plot
fig, axes = plt.subplots(1,3, figsize=(15,5))
for ax,(probs,title,color) in zip(axes,[
    (mean_p,    f"Uncalibrated (ECE={ece_raw:.3f})",    "#E24B4A"),
    (temp_probs,f"Temp Scaling (ECE={ece_temp:.3f})",   "#E88A2A"),
    (iso_probs, f"Isotonic Reg (ECE={ece_iso:.3f})",    "#639922"),
]):
    fp,mp = calibration_curve(true_labels, probs, n_bins=10)
    ax.plot([0,1],[0,1],"k--",alpha=0.5,label="Perfect")
    ax.plot(mp,fp,"o-",color=color,lw=2,label="Model")
    ax.fill_between(mp,mp,fp,alpha=0.1,color=color)
    ax.set_xlabel("Mean predicted confidence")
    ax.set_ylabel("Fraction positives")
    ax.set_title(title,fontweight="bold")
    ax.legend(); ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.suptitle("Three-Way Calibration — NIH ChestX-ray14",fontsize=13,fontweight="bold")
plt.tight_layout(); plt.savefig("calibration_three_way.png",dpi=150); plt.show()

## Cell 11 — Uncertainty vs accuracy (validation of MC signal)
With NIH's harder cases, uncertainty should now correctly rank case difficulty.

In [ ]:
preds_opt = (mean_p > best_thresh).astype(int)
bin_edges = np.percentile(var_p, np.linspace(0,100,6))
bin_accs, bin_lbls = [], []
for i in range(len(bin_edges)-1):
    lo,hi = bin_edges[i],bin_edges[i+1]
    mask  = (var_p>=lo)&(var_p<=hi) if i==len(bin_edges)-2 else (var_p>=lo)&(var_p<hi)
    if mask.sum()==0: continue
    bin_accs.append((preds_opt[mask]==true_labels[mask]).mean())
    bin_lbls.append(f"Q{i+1}\n(n={mask.sum()})")

fig,ax = plt.subplots(figsize=(7,4))
colors = ["#185FA5" if a>=0.80 else "#E88A2A" if a>=0.65 else "#E24B4A" for a in bin_accs]
bars   = ax.bar(bin_lbls, bin_accs, color=colors, alpha=0.85, edgecolor="white")
ax.axhline(0.5,color="red",linestyle="--",lw=1,label="Chance")
ax.axhline(np.mean(bin_accs),color="grey",linestyle=":",lw=1,
           label=f"Mean={np.mean(bin_accs):.2f}")
ax.set_xlabel("Uncertainty quintile (Q1=most confident, Q5=least confident)")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy vs MC Dropout Uncertainty — NIH dataset\n(should decrease Q1→Q5)")
ax.set_ylim(0,1.08); ax.legend()
for bar,acc in zip(bars,bin_accs):
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,
            f"{acc:.2f}",ha="center",va="bottom",fontsize=10,fontweight="bold")
plt.tight_layout(); plt.savefig("uncertainty_vs_accuracy.png",dpi=150); plt.show()

print(f"\nUncertainty correct   : {var_p[preds_opt==true_labels].mean():.6f}")
print(f"Uncertainty incorrect : {var_p[preds_opt!=true_labels].mean():.6f}")
ratio = var_p[preds_opt!=true_labels].mean() / (var_p[preds_opt==true_labels].mean()+1e-10)
print(f"Ratio (>1 = signal works): {ratio:.2f}x")

## Cell 12 — Clinical Triage Simulation ⭐
The headline contribution. With a working uncertainty signal on NIH,
high-uncertainty cases should correspond to genuinely difficult images.
Deferring them should meaningfully improve accuracy on the remainder.

In [ ]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix

sorted_by_unc = np.argsort(var_p)
defer_rates   = np.arange(0, 0.55, 0.05)
results = []
for dr in defer_rates:
    n_keep = int(len(var_p)*(1-dr))
    keep   = sorted_by_unc[:n_keep]
    if len(keep)==0: continue
    y_true,y_pred,y_prob = true_labels[keep],preds_opt[keep],mean_p[keep]
    acc  = (y_pred==y_true).mean()
    prec = precision_score(y_true,y_pred,zero_division=0)
    rec  = recall_score(y_true,y_pred,zero_division=0)
    spec = (y_pred[y_true==0]==0).mean() if (y_true==0).sum()>0 else 0
    auc  = roc_auc_score(y_true,y_prob) if len(np.unique(y_true))>1 else 0
    results.append(dict(deferral=dr,n_decided=n_keep,
        n_deferred=len(var_p)-n_keep,accuracy=acc,
        precision=prec,recall_pneu=rec,specificity=spec,auc=auc))

dr_v  = [r["deferral"]    for r in results]
ac_v  = [r["accuracy"]    for r in results]
au_v  = [r["auc"]         for r in results]
re_v  = [r["recall_pneu"] for r in results]
sp_v  = [r["specificity"] for r in results]

fig,axes = plt.subplots(1,2,figsize=(14,5))
ax=axes[0]
ax.plot([d*100 for d in dr_v],[a*100 for a in ac_v],"o-",
        color="#185FA5",lw=2.5,ms=7,label="Accuracy")
ax.plot([d*100 for d in dr_v],[a*100 for a in au_v],"s--",
        color="#639922",lw=2,ms=6,label="AUC-ROC")
ax.axhline(ac_v[0]*100,color="grey",linestyle=":",lw=1,
           label=f"Baseline = {ac_v[0]*100:.1f}%")
ax.fill_between([d*100 for d in dr_v],ac_v[0]*100,
                [a*100 for a in ac_v],alpha=0.1,color="#185FA5")
ax.set_xlabel("Cases deferred to radiologist (%)"); ax.set_ylabel("Performance (%)")
ax.set_title("Accuracy gain from uncertainty-based deferral",fontsize=11)
ax.legend(); ax.set_ylim(60,102); ax.grid(alpha=0.3)

ax=axes[1]
ax.plot([d*100 for d in dr_v],[r*100 for r in re_v],"o-",
        color="#E24B4A",lw=2.5,ms=7,label="Pneumonia recall")
ax.plot([d*100 for d in dr_v],[s*100 for s in sp_v],"s-",
        color="#185FA5",lw=2.5,ms=7,label="Normal specificity")
ax.set_xlabel("Cases deferred (%)"); ax.set_ylabel("Rate (%)")
ax.set_title("Sensitivity / Specificity vs deferral rate",fontsize=11)
ax.legend(); ax.set_ylim(50,102); ax.grid(alpha=0.3)

plt.suptitle("Clinical Triage Simulation — NIH ChestX-ray14",
             fontsize=13,fontweight="bold")
plt.tight_layout(); plt.savefig("triage_simulation.png",dpi=150); plt.show()

## Cell 13 — Triage summary table

In [ ]:
print("\n"+"="*80)
print(f"{'Deferral':>10} {'Decided':>8} {'Deferred':>9} {'Accuracy':>10} "
      f"{'Pneu Recall':>12} {'Specificity':>12} {'AUC':>8}")
print("="*80)
for r in results:
    flag=" ◄" if r["deferral"] in [0.0,0.10,0.20,0.30] else ""
    print(f"  {r['deferral']*100:>6.0f}%  {r['n_decided']:>8}  {r['n_deferred']:>8}  "
          f"  {r['accuracy']*100:>7.1f}%   {r['recall_pneu']*100:>9.1f}%   "
          f"{r['specificity']*100:>9.1f}%  {r['auc']:>6.3f}{flag}")
print("="*80)
r0  = results[0]
r10 = next(r for r in results if abs(r["deferral"]-0.10)<0.01)
r20 = next(r for r in results if abs(r["deferral"]-0.20)<0.01)
print(f"\nKEY FINDINGS:")
print(f"• Baseline (0% deferral)  : acc={r0['accuracy']*100:.1f}%  AUC={r0['auc']:.3f}")
print(f"• 10% deferral            : acc={r10['accuracy']*100:.1f}%  "
      f"(+{(r10['accuracy']-r0['accuracy'])*100:.1f}pp)")
print(f"• 20% deferral            : acc={r20['accuracy']*100:.1f}%  "
      f"(+{(r20['accuracy']-r0['accuracy'])*100:.1f}pp)")
print(f"• ECE: {ece_raw:.3f} → {ece_temp:.3f} (temp) → {ece_iso:.3f} (isotonic)")

## Cell 14 — Confusion matrices: baseline vs 20% deferral

In [ ]:
import itertools

def plot_cm(ax,cm,title):
    ax.imshow(cm,cmap="Blues",interpolation="nearest")
    ax.set_title(title,fontweight="bold",fontsize=12)
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(["Normal","Pneumonia"])
    ax.set_yticklabels(["Normal","Pneumonia"])
    thresh=cm.max()/2
    for i,j in itertools.product(range(2),range(2)):
        ax.text(j,i,f"{cm[i,j]}",ha="center",va="center",fontsize=14,
                color="white" if cm[i,j]>thresh else "black")
    ax.set_ylabel("True label"); ax.set_xlabel("Predicted label")

keep_20 = sorted_by_unc[:int(len(var_p)*0.80)]
acc_20  = (preds_opt[keep_20]==true_labels[keep_20]).mean()

fig,axes = plt.subplots(1,2,figsize=(12,5))
plot_cm(axes[0], confusion_matrix(true_labels,preds_opt),
        f"No deferral (n={len(true_labels)})\nAcc={ac_v[0]*100:.1f}%")
plot_cm(axes[1], confusion_matrix(true_labels[keep_20],preds_opt[keep_20]),
        f"20% deferred (n={len(keep_20)})\nAcc={acc_20*100:.1f}%")
plt.suptitle("Confusion Matrices — Effect of Uncertainty-Based Deferral",
             fontsize=13,fontweight="bold")
plt.tight_layout(); plt.savefig("confusion_matrices.png",dpi=150); plt.show()

## Cell 15 — Final metrics summary

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, classification_report

print("="*60)
print("FINAL RESULTS SUMMARY — NIH ChestX-ray14")
print("="*60)
print(f"AUC-ROC (MC mean)          : {roc_auc_score(true_labels,mean_p):.4f}")
print(f"AUC-ROC (isotonic)         : {roc_auc_score(true_labels,iso_probs):.4f}")
print(f"ECE uncalibrated           : {ece_raw:.4f}")
print(f"ECE temperature scaling    : {ece_temp:.4f}  (T={T_learned:.3f})")
print(f"ECE isotonic regression    : {ece_iso:.4f}")
print(f"Optimal threshold          : {best_thresh:.3f}")
print(f"Mean uncertainty (all)     : {var_p.mean():.5f}")
print(f"Mean uncertainty (correct) : {var_p[preds_opt==true_labels].mean():.5f}")
print(f"Mean uncertainty (wrong)   : {var_p[preds_opt!=true_labels].mean():.5f}")
print()
print("Classification report (optimal threshold):")
print(classification_report(true_labels,preds_opt,
    target_names=["Normal","Pneumonia"]))

## Cell 16 — Grad-CAM
Hooks into denseblock4 (same as v4).
With NIH's harder cases, expect high-uncertainty images to show
diffuse or off-target heatmaps vs focused lung-region activation
on low-uncertainty cases — this contrast is your Figure 4.

In [ ]:
import cv2

class GradCAM:
    def __init__(self,model,target_layer):
        self.model=model; self.gradients=None; self.activations=None
        target_layer.register_forward_hook(
            lambda m,i,o: setattr(self,"activations",o))
        target_layer.register_full_backward_hook(
            lambda m,gi,go: setattr(self,"gradients",go[0]))

    def generate(self,img_tensor):
        enable_mc_dropout(self.model)
        out=self.model(img_tensor); self.model.zero_grad()
        out[0,0].backward()
        w=self.gradients.mean(dim=[2,3],keepdim=True)
        cam=(w*self.activations).sum(dim=1).squeeze()
        cam=torch.relu(cam).cpu().detach().numpy()
        return (cam-cam.min())/(cam.max()-cam.min()+1e-8)

def show_gradcam(img_t,img_raw,label,pred,unc,thresh):
    cam  =gcam.generate(img_t.unsqueeze(0).to(device))
    cam_r=cv2.resize(cam,(224,224))
    heat =cv2.applyColorMap((cam_r*255).astype(np.uint8),cv2.COLORMAP_JET)
    orig =(img_raw.permute(1,2,0).numpy()*255).astype(np.uint8)
    overlay=cv2.addWeighted(orig,0.6,heat,0.4,0)
    correct=int(pred>thresh)==label
    fig,ax=plt.subplots(1,2,figsize=(8,4))
    ax[0].imshow(orig,cmap="gray")
    ax[0].set_title(f"True: {'Pneumonia' if label else 'Normal'}")
    ax[1].imshow(overlay)
    ax[1].set_title(f"{'✓ CORRECT' if correct else '✗ WRONG'}"
                    f"  pred={pred:.2f}  unc={unc:.5f}")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()

gcam = GradCAM(model, model.block4)

# Build a raw (un-normalised) test dataset for display
raw_val_tf = transforms.Compose([
    transforms.Resize((224,224)), transforms.ToTensor()])
test_raw = NIHChestDataset(test_ds.df, IMG_DIR, raw_val_tf)

sorted_idx = np.argsort(var_p)
for gname,idxs in [("LOW uncertainty (confident)", sorted_idx[:3]),
                    ("HIGH uncertainty (ambiguous)", sorted_idx[-3:])]:
    print(f"\n{'='*52}\n  {gname}\n{'='*52}")
    for i in idxs:
        img_t, lbl = test_ds[i]
        img_raw, _ = test_raw[i]
        show_gradcam(img_t,img_raw,lbl,mean_p[i],var_p[i],best_thresh)